In [2]:
import os
from dotenv import load_dotenv
import openai
import re

load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")

SYSTEM_PROMPT = """
You are generating synthetic medical French transcripts for urgent care scenarios.
Write short clinical-style phrases in French, similar to how doctors dictate or write urgent care notes.
- Use common medical acronyms (like PEC, HD, HTA, DRA, DSM, etc.).
- Sentences should often be abbreviated, fragmented, or telegraphic (not fully grammatical).
- Cover topics like trauma, cardiac issues, respiratory problems, post-op complications, metabolic disorders, and urgent care evaluations.
- Mix single-word diagnoses and longer notes (~5-20 words).
- Use natural French medical shorthand.
Example style: "stable HD, pas de signe de DRA", "gsc 15, pas de DSM", "PEC SMUR, malaise au domicile".
"""

USER_PROMPT = """
Generate 1500 French urgent care medical phrases according to the above instructions.
Each phrase should be on a new line.
Vary between very short (1-2 words) and longer (~20 words) entries.
Do NOT number them.
Do NOT add any explanations.
Just output the phrases, one per line.
"""

def generate_corpus(system_prompt, user_prompt, model="gpt-4", temperature=0.7, max_tokens=3000):
    response = openai.ChatCompletion.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    text = response['choices'][0]['message']['content']

    # Remove any accidental numbering (just in case)
    lines = text.splitlines()
    cleaned_lines = []
    for line in lines:
        line = re.sub(r"^\s*\d+[\).\s-]+", "", line).strip()
        if line:
            cleaned_lines.append(line)
    
    return "\n".join(cleaned_lines)

output_path = "../data/synthetic_medical_corpus.txt"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

corpus = generate_corpus(SYSTEM_PROMPT, USER_PROMPT)

with open(output_path, "w", encoding="utf-8") as f:
    f.write(corpus)

print(f"✅ Synthetic corpus saved to {output_path}")

✅ Synthetic corpus saved to ../data/synthetic_medical_corpus.txt
